# gnss_gpu in 3 minutes — surviving the urban canyon

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/gnss_gpu/blob/main/examples/colab_urban_canyon_quickstart.ipynb)
&nbsp; [GitHub repo](https://github.com/rsasaki0109/gnss_gpu) · [Live results snapshot](https://rsasaki0109.github.io/gnss_gpu/)

**No GPU, no build, no downloaded data.** This notebook runs entirely on the
Colab CPU with NumPy and the pure-Python solver that ships with
[`gnss_gpu`](https://github.com/rsasaki0109/gnss_gpu).

**What you'll see.** In a dense city, tall buildings block the direct path to
low-elevation satellites, so the receiver locks onto *reflected* signals
instead. Those reflections travel farther and inflate the measured range by
tens of metres (NLOS multipath). A plain least-squares fix trusts every
satellite equally and gets dragged off the road. We simulate that exact
situation along a short driving segment and solve every epoch two ways **with
the same inputs**:

| method | idea |
|---|---|
| naive WLS (L2) | ordinary least squares — every satellite trusted equally |
| robust SPP (Cauchy) | IRLS that softly down-weights large-residual (NLOS-biased) satellites |

The robust solver is the real package code (`gnss_gpu.robust_spp`); only the
measurement environment is simulated. Robust down-weighting is the core idea
the GPU particle-filter stack scales up to beat RTKLIB demo5 on real UrbanNav
data (see the [README](https://github.com/rsasaki0109/gnss_gpu#readme)).

In [ ]:
# Setup: grab the repo and import the solver (pure Python + NumPy, ~10 s).
import os
import subprocess
import sys

try:
    from gnss_gpu.robust_spp import robust_spp
except ImportError:
    local_pkg = os.path.abspath(os.path.join("..", "python"))
    if os.path.isdir(os.path.join(local_pkg, "gnss_gpu")):
        # Running inside a local checkout (examples/) — no clone needed.
        sys.path.insert(0, local_pkg)
    else:
        # Colab / fresh environment: shallow-clone the repo.
        if not os.path.isdir("gnss_gpu"):
            subprocess.run(
                ["git", "clone", "--depth", "1",
                 "https://github.com/rsasaki0109/gnss_gpu.git"],
                check=True,
            )
        sys.path.insert(0, os.path.abspath(os.path.join("gnss_gpu", "python")))
    # Drop any stale namespace-package cache (the cloned repo dir shadows the
    # package name when the cwd is on sys.path, e.g. after a kernel restart).
    sys.modules.pop("gnss_gpu", None)
    from gnss_gpu.robust_spp import robust_spp

print("gnss_gpu robust SPP solver ready — no GPU, no build, no data needed.")

## 1. Build the scene

A receiver drives east through central Tokyo at ~18 km/h for 60 seconds.
Thirteen satellites are visible: ten high-elevation ones with a clean line of
sight, and three low-elevation ones whose direct path the buildings block —
each of those carries a +30–80 m NLOS range bias from the reflected path.

In [ ]:
import math

import numpy as np

# WGS84 constants
WGS84_A = 6378137.0
WGS84_F = 1.0 / 298.257223563
WGS84_E2 = 2 * WGS84_F - WGS84_F * WGS84_F


def llh_to_ecef(lat_deg, lon_deg, alt_m):
    """Geodetic latitude/longitude/height -> ECEF position [m]."""
    lat, lon = math.radians(lat_deg), math.radians(lon_deg)
    sin_lat = math.sin(lat)
    n = WGS84_A / math.sqrt(1.0 - WGS84_E2 * sin_lat * sin_lat)
    return np.array([
        (n + alt_m) * math.cos(lat) * math.cos(lon),
        (n + alt_m) * math.cos(lat) * math.sin(lon),
        (n * (1.0 - WGS84_E2) + alt_m) * sin_lat,
    ])


def enu_to_ecef_matrix(lat_deg, lon_deg):
    """Columns are the local East/North/Up axes expressed in ECEF."""
    lat, lon = math.radians(lat_deg), math.radians(lon_deg)
    s_lat, c_lat = math.sin(lat), math.cos(lat)
    s_lon, c_lon = math.sin(lon), math.cos(lon)
    return np.column_stack([
        [-s_lon, c_lon, 0.0],
        [-s_lat * c_lon, -s_lat * s_lon, c_lat],
        [c_lat * c_lon, c_lat * s_lon, s_lat],
    ])


def build_constellation(rx_ecef, enu, sats_azel_deg):
    """Place satellites at a typical MEO slant range along given (az, el)."""
    slant_range_m = 21_000_000.0
    sats = np.empty((len(sats_azel_deg), 3))
    for i, (az_deg, el_deg) in enumerate(sats_azel_deg):
        az, el = math.radians(az_deg), math.radians(el_deg)
        los_enu = np.array(
            [math.cos(el) * math.sin(az), math.cos(el) * math.cos(az), math.sin(el)]
        )
        sats[i] = rx_ecef + slant_range_m * (enu @ los_enu)
    return sats


def geometric_ranges(rx_ecef, sats_ecef):
    """Euclidean receiver-to-satellite ranges [m] (matches robust_spp's model)."""
    return np.linalg.norm(sats_ecef - rx_ecef, axis=1)


rng = np.random.default_rng(20260525)

start_lat, start_lon, start_alt = 35.6804, 139.7690, 45.0  # central Tokyo
enu = enu_to_ecef_matrix(start_lat, start_lon)
east_axis_ecef = enu @ np.array([1.0, 0.0, 0.0])
start_ecef = llh_to_ecef(start_lat, start_lon, start_alt)

# Fixed sky geometry: (azimuth from north [deg], elevation [deg]).
sats_azel = [
    (25.0, 70.0), (80.0, 62.0), (150.0, 55.0), (210.0, 68.0), (290.0, 58.0),
    (330.0, 48.0), (45.0, 42.0), (110.0, 38.0), (180.0, 45.0), (260.0, 40.0),
    (70.0, 15.0), (200.0, 18.0), (310.0, 12.0),
]
sats_ecef = build_constellation(start_ecef, enu, sats_azel)
elevations = np.array([el for _, el in sats_azel])

# Buildings block low-elevation satellites -> positive NLOS range bias.
nlos_mask = elevations < 30.0
nlos_bias_m = np.where(
    nlos_mask, rng.uniform(30.0, 80.0, size=len(elevations)), 0.0
)

print(
    f"{len(sats_azel)} satellites: {int(np.sum(~nlos_mask))} clean, "
    f"{int(np.sum(nlos_mask))} NLOS-blocked "
    f"(+{nlos_bias_m[nlos_mask].min():.0f} to "
    f"+{nlos_bias_m[nlos_mask].max():.0f} m range bias)"
)

In [ ]:
# Sky plot: which satellites does the city block?
import matplotlib.pyplot as plt

az_rad = np.radians([az for az, _ in sats_azel])
zenith_dist = 90.0 - elevations  # polar radius: 0 = zenith, 90 = horizon

fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw={"projection": "polar"})
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
ax.scatter(
    az_rad[~nlos_mask], zenith_dist[~nlos_mask],
    s=150, c="tab:green", marker="o", label="clean line of sight",
)
ax.scatter(
    az_rad[nlos_mask], zenith_dist[nlos_mask],
    s=180, c="tab:red", marker="X", label="NLOS-blocked (+30-80 m bias)",
)
ax.set_rlim(0, 90)
ax.set_rticks([30, 60, 90])
ax.set_yticklabels(["60°", "30°", "0° elev"])
ax.set_title("Buildings block the low-elevation satellites", pad=18)
ax.legend(loc="upper left", bbox_to_anchor=(0.85, 1.1))
plt.tight_layout()
plt.show()

## 2. Solve every epoch twice — same inputs, same solver

We reuse one IRLS solver for both methods so the comparison isolates the robust
*weighting*, not implementation differences: a huge kernel threshold makes the
robust weights inert (recovering plain least squares), while a 15 m Cauchy
threshold softly down-weights large-residual satellites.

In [ ]:
# A generous threshold makes the kernel inert -> plain least squares.
PLAIN_LS_THRESHOLD_M = 1.0e12
ROBUST_THRESHOLD_M = 15.0

n_epochs = 60          # 60 s at 1 Hz
speed_mps = 5.0        # ~18 km/h city driving
true_clock_bias_m = 1234.5
code_noise_sigma_m = 1.5
init_guess = start_ecef + np.array([40.0, -30.0, 20.0])  # coarse shared prior


def to_east_north(p_ecef):
    d = enu.T @ (p_ecef - start_ecef)
    return d[:2]


truth_en, naive_en, robust_en = [], [], []
for k in range(n_epochs):
    rx_true = start_ecef + speed_mps * k * east_axis_ecef
    noise = rng.normal(0.0, code_noise_sigma_m, size=len(sats_ecef))
    pseudoranges = (
        geometric_ranges(rx_true, sats_ecef)
        + true_clock_bias_m + noise + nlos_bias_m
    )
    naive = robust_spp(
        sats_ecef, pseudoranges, init_pos=init_guess,
        weight_func="huber", threshold=PLAIN_LS_THRESHOLD_M,
    )
    robust = robust_spp(
        sats_ecef, pseudoranges, init_pos=init_guess,
        weight_func="cauchy", threshold=ROBUST_THRESHOLD_M,
    )
    truth_en.append(to_east_north(rx_true))
    naive_en.append(to_east_north(naive))
    robust_en.append(to_east_north(robust))

truth_en = np.array(truth_en)
naive_en = np.array(naive_en)
robust_en = np.array(robust_en)
naive_errors = np.linalg.norm(naive_en - truth_en, axis=1)
robust_errors = np.linalg.norm(robust_en - truth_en, axis=1)

naive_p50 = float(np.median(naive_errors))
naive_rms = float(np.sqrt(np.mean(naive_errors**2)))
robust_p50 = float(np.median(robust_errors))
robust_rms = float(np.sqrt(np.mean(robust_errors**2)))
wins = int(np.sum(robust_errors < naive_errors))

print(f"{'method':<26}{'P50 err':>12}{'RMS err':>12}")
print("-" * 50)
print(f"{'naive WLS (L2)':<26}{naive_p50:>10.2f} m{naive_rms:>10.2f} m")
print(f"{'robust SPP (Cauchy)':<26}{robust_p50:>10.2f} m{robust_rms:>10.2f} m")
print("-" * 50)
print(
    f"robust vs naive: {100 * (1 - robust_p50 / naive_p50):.0f}% better P50, "
    f"{100 * (1 - robust_rms / naive_rms):.0f}% better RMS; "
    f"robust closer to truth in {wins}/{n_epochs} epochs"
)

In [ ]:
# Trajectory: NLOS bias drags the naive fix off the road.
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(truth_en[:, 0], truth_en[:, 1], "k-", lw=2.5, label="truth (driving east)")
ax.plot(
    naive_en[:, 0], naive_en[:, 1], "o-", c="tab:orange", ms=3.5, lw=1,
    alpha=0.85, label=f"naive WLS — P50 {naive_p50:.2f} m",
)
ax.plot(
    robust_en[:, 0], robust_en[:, 1], "o-", c="tab:blue", ms=3.5, lw=1,
    alpha=0.85, label=f"robust SPP — P50 {robust_p50:.2f} m",
)
ax.set_xlabel("East [m]")
ax.set_ylabel("North [m]")
ax.set_aspect("equal")
ax.grid(alpha=0.3)
ax.legend(loc="center left")
ax.set_title("Same measurements, two solvers: the robust fix stays on the road")
plt.tight_layout()
plt.show()

In [ ]:
# Horizontal error over time.
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(naive_errors, c="tab:orange", lw=1.5, label="naive WLS (L2)")
ax.plot(robust_errors, c="tab:blue", lw=1.5, label="robust SPP (Cauchy)")
ax.set_xlabel("epoch [s]")
ax.set_ylabel("horizontal error [m]")
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend()
ax.set_title("Per-epoch horizontal error")
plt.tight_layout()
plt.show()

## Takeaway — and where this goes next

The Cauchy kernel softly down-weights the large-residual (NLOS-biased)
pseudoranges, so they pull the fix less and it stays close to truth — here
**~80% better P50** from a one-line change of weighting.

Scaling this robust-weighting idea up — 100K–1M GPU particles, ray-traced
line-of-sight checks against real 3D building meshes, double-difference
carrier tracking, factor-graph optimization — is what lets the full stack beat
RTKLIB demo5 on real UrbanNav urban driving data
(**1.36 m vs 2.67 m P50** on Tokyo Odaiba):

- 📊 [Accuracy results & honest comparisons](https://github.com/rsasaki0109/gnss_gpu#results-at-a-glance) · [GPU throughput benchmarks](https://github.com/rsasaki0109/gnss_gpu/blob/main/benchmarks/RESULTS.md)
- 🛰️ [More demos (GPU pipeline, RINEX, PLATEAU NLOS ray tracing)](https://github.com/rsasaki0109/gnss_gpu/tree/main/examples)
- 🌐 [Live results snapshot](https://rsasaki0109.github.io/gnss_gpu/)

If this was useful, a ⭐ on
[**rsasaki0109/gnss_gpu**](https://github.com/rsasaki0109/gnss_gpu)
helps others find it.